# Day 2 Lab 2 :
## 做一个属于你的 AI 职业数字人（可控版）

今天我们要做一件很有意思的事情：

👉 把一份简历，变成一个“会说话的人”

并且，让它：

- 不乱说  
- 有边界  
- 更像一个真实的人  

---

我们会做三件事：

1. 让 AI 成为这个人  
2. 限制它不回答不该回答的问题  
3. 让另一个 AI 来检查它

## 🧰 Step 1：准备工具

在开始之前，我们先做一件工程师最重要的事：

👉 **把数据读进来**

---

在这个项目里，我们需要做两件事：

- 📄 读取 PDF 简历  
- 🖥️ 优化输出展示（让结果更清晰）

---

💡 小提示：

在 Notebook 里：

- `print()` 更像调试工具  
- `display()` 更适合展示内容  

接下来，我们先把这些工具准备好

In [ ]:
# ===== Cell 1: 准备工具 =====

# 如果你还没安装，请先运行：
# !pip install pypdf python-dotenv

from IPython.display import display, Markdown
from pypdf import PdfReader

def read_pdf(file_path):
    """
    读取 PDF 文件中的全部文本
    """
    reader = PdfReader(file_path)
    text = ""
    
    for i, page in enumerate(reader.pages):
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
        else:
            print(f"⚠️ 第{i+1}页没有读取到文本")
    
    return text

## Step 2：准备大语言模型

接下来，我们连接大模型。

你可以把它理解成：

👉 一个“可以对话的引擎”

这里有两个角色：

- system：决定它是谁  
- user：决定我们问什么  

今天，我们会用 system，让 AI “变成一个人”

In [ ]:
# ===== Cell 2: 准备大语言模型（千问版本） =====

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# 创建客户端（通义千问）
client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("API_KEY")
)

def call_llm(prompt, system_prompt):
    """
    最简大模型调用函数
    """
    response = client.chat.completions.create(
        model="qwen-plus",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    
    return response.choices[0].message.content

## Step 3：读取简历

先确认一件事：

👉 简历有没有真的被读进来

In [ ]:
# ===== Cell 3: 读取简历 =====

resume_text = read_pdf("职业数字人示例简历.pdf")

display(Markdown("简历内容预览：\n"))
display(Markdown(resume_text[:500]))

## 👤 Step 4：让 AI 成为这个人

现在，我们来做一件关键的事情：

👉 **让 AI 不再“介绍他”，而是“成为他”**

---

在 Lab 1 里，我们只用到了一个角色：

- user：我们问问题  

但今天，我们多加了一层：

- system：用来告诉 AI “它是谁”

---

💡 关键点：

👉 system 决定“人格”  
👉 user 决定“问题”

---

接下来，我们会做一件非常简单但很神奇的事情：

👉 只改一段 system prompt

让 AI：

- 用第一人称回答（“我…”）  
- 只基于简历内容  
- 更像一个真实的人  

---

🎯 你可以关注一个变化：

👉 同一个问题  
👉 不同的 system  

AI的回答会完全不一样

In [ ]:
#==== Cell 4.1: 设置人设并提问 =====
system_prompt = f"""
你是这份简历中的本人。

请用第一人称回答问题，语气自然，像真实的人。

规则：
- 只能基于简历中的信息回答
- 如果简历中没有明确提到：
  不要直接拒绝
  请用“委婉、不确定”的方式表达，例如：
    - “从我的经历来看…”
    - “虽然简历中没有明确写到…但…”
    - “我目前更多的经验还是在…方面”

- 不要编造具体事实（例如没有写的经历、职位、成果）

简历如下：
{resume_text}
"""

In [ ]:
# ===== Cell 4.2: 对比——AI助手 vs 数字人 =====

question = "简短说明，你做过哪些项目？"

# ① 普通AI助手
answer1 = call_llm(question, "你是一个AI助手")

# ② 职业数字人
answer2 = call_llm(question, system_prompt)

display(Markdown("【AI助手的回答】\n"))
display(Markdown(answer1))

display(Markdown("\n" + "="*50 + "\n"))

display(Markdown("【职业数字人的回答】\n"))
display(Markdown(answer2))

## 🛑 Step 5：先控制输入范围

现在，数字人已经能“像这个人一样说话”了。

但问题来了：

👉 有些问题，其实不应该让它回答。

比如：

- 隐私信息（邮箱、电话、住址）
- 其他明显不适合公开回答的内容

---

所以在真正的系统里，我们通常会先做一层很简单的输入控制：

👉 **先判断这个问题能不能进来**

---

这一层不一定需要 AI。

很多时候，用最简单的规则就够了。

---

下面我们做一个对比：

- 不加输入控制，会发生什么？
- 加上输入控制之后，会发生什么？

In [ ]:
# ===== Cell 5.1: 不加输入控制 =====

question = "他的邮箱是什么？"

answer = call_llm(question, system_prompt)

display(Markdown("### ❓ 用户问题"))
display(Markdown(question))

display(Markdown("---"))

display(Markdown("### 🤖 不加输入控制时，AI的回答"))
display(Markdown(answer))

In [ ]:
# ===== Cell 5.2: 加上输入 Guardrail =====

def input_guardrail(question):
    forbidden_words = ["薪资", "电话", "邮箱", "微信", "住址", "家庭", "年龄"]

    for word in forbidden_words:
        if word in question:
            return False, "这个问题涉及隐私信息，我不方便回答。"

    return True, "ok"


question = "他的邮箱是什么？"

valid, msg = input_guardrail(question)

display(Markdown("### ❓ 用户问题"))
display(Markdown(question))

display(Markdown("---"))

if not valid:
    display(Markdown("### 🛡️ 加上输入 Guardrail 后"))
    display(Markdown(msg))
else:
    answer = call_llm(question, system_prompt)
    display(Markdown("### 🤖 AI回答"))
    display(Markdown(answer))

## Step 6：让 AI 检查 AI（Evaluator）

即使我们做了限制，仍然有一个问题：

👉 AI 会不会“合理发挥”？

所以我们再加一层：

👉 让另一个 AI 来检查它

就像：

写完答案之后，有人帮你审稿

In [ ]:

# ===== Cell 6: Evaluator =====

def evaluator(question, answer, resume_text):
    eval_prompt = f"""
请你作为“职业数字人审核员”，检查下面这段回答是否合格。

你的审核标准有两个：

【1. 人设是否自然】
请判断这段回答是否像一个真实的职场人会说的话。
重点看：
- 是否自然
- 是否有正常的职业发展逻辑
- 是否体现出一个人希望成长、升职、加薪、承担更多价值的思考方式
- 是否过于机械、像AI模板
- 是否太空、太假、太官话

【2. 内容是否越界】
请判断这段回答是否超出了简历依据。
重点看：
- 是否编造了明确事实
- 是否把推测说成了事实
- 是否出现了简历中没有依据的经历、职责、成果

【简历内容】
{resume_text}

【用户问题】
{question}

【AI回答】
{answer}

请按下面格式输出：

是否自然：PASS 或 FAIL
自然性点评：...

是否越界：PASS 或 FAIL
越界点评：...

最终结论：PASS 或 FAIL
修改建议：如果需要，请给出一个更自然、更安全的版本
"""
    return call_llm(eval_prompt, "你是一个严格但懂职场表达的审核员。")

In [ ]:
#==== Cell 6.2: 测试 Evaluator =====
question = "你觉得自己是否具备管理能力？请具体说明"

answer = call_llm(question, system_prompt)
review = evaluator(question, answer, resume_text)

display(Markdown("### ❓ 用户问题"))
display(Markdown(question))

display(Markdown("---"))

display(Markdown("### 👤 职业数字人的回答"))
display(Markdown(answer))

display(Markdown("---"))

display(Markdown("### 🔍 Evaluator 审核结果"))
display(Markdown(review))

## 总结

今天我们做的，不只是一个聊天机器人

我们做的是一个更接近真实产品的 AI：

👉 一个职业数字人

并且，我们加了三件很重要的东西：

1. 让它成为这个人  
2. 不回答不该回答的问题  
3. 用另一个 AI 来检查它  

---

一个好的 AI，不只是聪明，

👉 更重要的是：可控